In [1]:
import pandas as pd
import numpy as np
from google.colab import files

# Path of the uploaded dirty dataset
input_path = "/content/sample_data/dirty_dataset.csv"

# Load the original dataset
df = pd.read_csv(input_path)

# Always clean a copy
df_clean = df.copy()

print("Dataset loaded successfully.")
print("Original shape:", df_clean.shape)

display(df_clean.head())

Dataset loaded successfully.
Original shape: (20300, 15)


,employee_id,name,age,salary,join_date,department,gender,country,city,weight_kg,is_active,review,price,weight_kg_duplicate,target
0,12369,Hasan D.,200.0,35149,2023-05-30,SALES,Man,America,Sylhet,118.4,0,Good value for money.,219.18,NaN,0
1,7361,Grace P.,NaN,77836,2017-02-27,Engineering,FEMALE,America,Chittagong,71kg,True,Product met all expectations.,329.26,NaN,0
2,3446,Eve D.,20.0,35404,2022-01-18,hr,male,BANGLADESH,Chitagong,98.7,False,Very satisfied with the quality.,438.6,NaN,0
3,18416,Diana E.,47.0,107375,2017-04-15,Marketing,Male,U.K.,Khulna,NaN,false,Delivery was fast and packaging great.,238.03,NaN,0
4,13677,Ira W.,64.0,55962,2021-11-02,SALES,F,United Kingdom,Dhaka,83.5,false,Would buy again without hesitation.,9.4,NaN,0


In [2]:
print("Column names:")
print(df_clean.columns.tolist())

print("\nData types:")
print(df_clean.dtypes)

print("\nMissing values:")
print(df_clean.isnull().sum())

print("\nExact duplicate rows:")
print(df_clean.duplicated().sum())

print("\nDuplicate employee IDs:")
print(df_clean["employee_id"].duplicated().sum())

Column names:
['employee_id', 'name', 'age', 'salary', 'join_date', 'department', 'gender', 'country', 'city', 'weight_kg', 'is_active', 'review', 'price', 'weight_kg_duplicate', 'target']

Data types:
employee_id              int64
name                    object
age                    float64
salary                  object
join_date               object
department              object
gender                  object
country                 object
city                    object
weight_kg               object
is_active               object
review                  object
price                   object
weight_kg_duplicate    float64
target                   int64
dtype: object

Missing values:
employee_id                0
name                       0
age                      621
salary                   610
join_date                810
department               857
gender                  1632
country                  608
city                     604
weight_kg                643
is_active   

In [4]:
# Find all text columns
object_columns = df_clean.select_dtypes(include="object").columns

# Remove leading and trailing spaces
for column in object_columns:
    df_clean[column] = (
        df_clean[column]
        .astype("string")
        .str.strip()
    )

# Convert empty strings into missing values
df_clean[object_columns] = df_clean[object_columns].replace(
    "",
    pd.NA
)

print("Empty strings and unnecessary spaces normalized.")

Empty strings and unnecessary spaces normalized.


In [5]:
shape_before = df_clean.shape
duplicates_before = df_clean.duplicated().sum()

df_clean = df_clean.drop_duplicates().copy()

shape_after = df_clean.shape

print("Duplicate rows before:", duplicates_before)
print("Shape before:", shape_before)
print("Shape after:", shape_after)
print("Rows removed:", shape_before[0] - shape_after[0])

Duplicate rows before: 300
Shape before: (20300, 15)
Shape after: (20000, 15)
Rows removed: 300


In [6]:
duplicate_ids_before = df_clean["employee_id"].duplicated().sum()

print("Duplicate employee IDs before:", duplicate_ids_before)

# Keep the first record for each employee ID
df_clean = df_clean.drop_duplicates(
    subset=["employee_id"],
    keep="first"
).copy()

duplicate_ids_after = df_clean["employee_id"].duplicated().sum()

print("Duplicate employee IDs after:", duplicate_ids_after)
print("Current shape:", df_clean.shape)

Duplicate employee IDs before: 199
Duplicate employee IDs after: 0
Current shape: (19801, 15)


In [7]:
def clean_numeric_column(series, remove_tokens=()):
    """
    Remove unwanted symbols and convert a column to float.
    Invalid values become NaN.
    """
    cleaned = series.astype("string").str.strip()

    for token in remove_tokens:
        cleaned = cleaned.str.replace(
            token,
            "",
            regex=False
        )

    return pd.to_numeric(
        cleaned,
        errors="coerce"
    ).astype("float64")


# Convert age
df_clean["age"] = clean_numeric_column(
    df_clean["age"]
)

# Remove $ and commas from salary
df_clean["salary"] = clean_numeric_column(
    df_clean["salary"],
    ["$", ","]
)

# Remove kg from weight
df_clean["weight_kg"] = clean_numeric_column(
    df_clean["weight_kg"],
    ["kg", "KG", "Kg"]
)

# Remove $ and commas from price
df_clean["price"] = clean_numeric_column(
    df_clean["price"],
    ["$", ","]
)

print(
    df_clean[
        ["age", "salary", "weight_kg", "price"]
    ].dtypes
)

age          float64
salary       float64
weight_kg    float64
price        float64
dtype: object


In [8]:
# Date formats found in the dataset
date_formats = [
    "%Y-%m-%d",
    "%Y/%m/%d",
    "%d/%m/%Y",
    "%m-%d-%Y",
    "%d-%b-%Y"
]


def parse_mixed_date(value):
    """
    Try every expected date format.
    Return NaT when a date cannot be converted.
    """
    if pd.isna(value):
        return pd.NaT

    value = str(value).strip()

    for date_format in date_formats:
        try:
            return pd.to_datetime(
                value,
                format=date_format
            )
        except (ValueError, TypeError):
            continue

    return pd.NaT


df_clean["join_date"] = df_clean["join_date"].apply(
    parse_mixed_date
)

print("Date column type:", df_clean["join_date"].dtype)
print(
    "Missing or unrecognized dates:",
    df_clean["join_date"].isnull().sum()
)

display(df_clean[["join_date"]].head())

Date column type: datetime64[ns]
Missing or unrecognized dates: 792


,join_date
0,2023-05-30
1,2017-02-27
2,2022-01-18
3,2017-04-15
4,2021-11-02


In [9]:
department_map = {
    "hr": "HR",
    "engineering": "Engineering",
    "sales": "Sales",
    "marketing": "Marketing",
    "mrketing": "Marketing",
    "finance": "Finance",
    "finace": "Finance"
}

df_clean["department"] = (
    df_clean["department"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map(department_map)
)

print(df_clean["department"].value_counts(dropna=False))

department
Sales          3815
HR             3805
Engineering    3794
Marketing      3788
Finance        3765
NaN             834
Name: count, dtype: int64


In [10]:
gender_map = {
    "male": "Male",
    "m": "Male",
    "man": "Male",

    "female": "Female",
    "f": "Female",
    "woman": "Female"
}

df_clean["gender"] = (
    df_clean["gender"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map(gender_map)
)

print(df_clean["gender"].value_counts(dropna=False))

gender
Male      9178
Female    9025
NaN       1598
Name: count, dtype: int64


In [11]:
country_map = {
    "usa": "United States",
    "u.s.a": "United States",
    "us": "United States",
    "america": "United States",
    "united states": "United States",

    "uk": "United Kingdom",
    "u.k.": "United Kingdom",
    "united kingdom": "United Kingdom",

    "bd": "Bangladesh",
    "bangladesh": "Bangladesh"
}

df_clean["country"] = (
    df_clean["country"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map(country_map)
)

print(df_clean["country"].value_counts(dropna=False))

country
United States     8924
Bangladesh        5887
United Kingdom    4402
NaN                588
Name: count, dtype: int64


In [12]:
city_map = {
    "dhka": "Dhaka",
    "dhakka": "Dhaka",
    "dhaka": "Dhaka",

    "chitagong": "Chittagong",
    "chittagongg": "Chittagong",
    "chittagong": "Chittagong",

    "slyhet": "Sylhet",
    "sylhet": "Sylhet",

    "rajshai": "Rajshahi",
    "rajshahi": "Rajshahi",

    "khulna": "Khulna"
}

df_clean["city"] = (
    df_clean["city"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map(city_map)
)

print(df_clean["city"].value_counts(dropna=False))

city
Dhaka         4050
Chittagong    4018
Rajshahi      3719
Khulna        3715
Sylhet        3710
NaN            589
Name: count, dtype: int64


In [13]:
boolean_map = {
    "true": True,
    "1": True,
    "false": False,
    "0": False
}

normalized_active = (
    df_clean["is_active"]
    .astype("string")
    .str.strip()
    .str.lower()
)

mapped_active = normalized_active.map(boolean_map)

# Use Pandas nullable Boolean type temporarily
df_clean["is_active"] = pd.array(
    mapped_active,
    dtype="boolean"
)

print(df_clean["is_active"].value_counts(dropna=False))

is_active
False    9654
True     9579
<NA>      568
Name: count, dtype: Int64


In [14]:
noisy_reviews = {
    "ok",
    "bad",
    "na",
    "n/a",
    "...",
    "fine",
    "not bad",
    "ok ok",
    ".",
    "good"
}

review_cleaned = (
    df_clean["review"]
    .astype("string")
    .str.strip()
)

# Replace noisy reviews with missing values
df_clean["review"] = review_cleaned.mask(
    review_cleaned.str.lower().isin(noisy_reviews),
    pd.NA
)

print(df_clean["review"].value_counts(dropna=False))

review
<NA>                                      3448
Delivery was fast and packaging great.    2768
Good value for money.                     2740
Product met all expectations.             2730
Excellent product, highly recommend!      2722
Would buy again without hesitation.       2701
Very satisfied with the quality.          2692
Name: count, dtype: Int64


In [15]:
# Valid age range: 18 to 65
invalid_age_mask = ~df_clean["age"].between(
    18,
    65
)

print("Invalid ages found:", invalid_age_mask.sum())

df_clean.loc[
    invalid_age_mask,
    "age"
] = np.nan


# Valid price range: greater than 0 and at most 10,000
invalid_price_mask = (
    (df_clean["price"] <= 0) |
    (df_clean["price"] > 10000)
)

print("Invalid prices found:", invalid_price_mask.sum())

df_clean.loc[
    invalid_price_mask,
    "price"
] = np.nan


# Salary must be at least 15,000
invalid_salary_mask = df_clean["salary"] < 15000

print("Invalid low or negative salaries:", invalid_salary_mask.sum())

df_clean.loc[
    invalid_salary_mask,
    "salary"
] = np.nan

Invalid ages found: 991
Invalid prices found: 390
Invalid low or negative salaries: 261


In [16]:
salary_q1 = df_clean["salary"].quantile(0.25)
salary_q3 = df_clean["salary"].quantile(0.75)

salary_iqr = salary_q3 - salary_q1

calculated_lower_bound = salary_q1 - 1.5 * salary_iqr
calculated_upper_bound = salary_q3 + 1.5 * salary_iqr

# Combine IQR limits with assignment business limits
salary_lower_bound = max(
    15000.0,
    calculated_lower_bound
)

salary_upper_bound = min(
    500000.0,
    calculated_upper_bound
)

salary_outlier_mask = (
    (df_clean["salary"] < salary_lower_bound) |
    (df_clean["salary"] > salary_upper_bound)
)

print("Salary Q1:", salary_q1)
print("Salary Q3:", salary_q3)
print("Salary IQR:", salary_iqr)
print("Final lower bound:", salary_lower_bound)
print("Final upper bound:", salary_upper_bound)
print("Salary outliers found:", salary_outlier_mask.sum())

# Cap the outliers
df_clean["salary"] = df_clean["salary"].clip(
    lower=salary_lower_bound,
    upper=salary_upper_bound
)

Salary Q1: 52365.75
Salary Q3: 97555.25
Salary IQR: 45189.5
Final lower bound: 15000.0
Final upper bound: 165339.5
Salary outliers found: 313


In [17]:
numeric_columns = [
    "age",
    "salary",
    "weight_kg",
    "price"
]

for column in numeric_columns:
    median_value = df_clean[column].median()

    print(
        f"{column} median used:",
        median_value
    )

    df_clean[column] = df_clean[column].fillna(
        median_value
    )

age median used: 42.0
salary median used: 75126.0
weight_kg median used: 86.6
price median used: 250.1


In [18]:
categorical_columns = [
    "department",
    "gender",
    "country",
    "city"
]

for column in categorical_columns:
    mode_value = df_clean[column].mode(
        dropna=True
    )[0]

    print(
        f"{column} mode used:",
        mode_value
    )

    df_clean[column] = df_clean[column].fillna(
        mode_value
    )

department mode used: Sales
gender mode used: Male
country mode used: United States
city mode used: Dhaka


In [19]:
valid_dates = (
    df_clean["join_date"]
    .dropna()
    .sort_values()
)

median_date = valid_dates.iloc[
    len(valid_dates) // 2
]

df_clean["join_date"] = df_clean["join_date"].fillna(
    median_date
)

print("Median date used:", median_date)
print(
    "Missing dates remaining:",
    df_clean["join_date"].isnull().sum()
)

Median date used: 2017-06-30 00:00:00
Missing dates remaining: 0


In [20]:
# Missing reviews become "No Review"
df_clean["review"] = df_clean["review"].fillna(
    "No Review"
)

# Missing active status becomes False
df_clean["is_active"] = (
    df_clean["is_active"]
    .fillna(False)
    .astype(bool)
)

print("Missing reviews:", df_clean["review"].isnull().sum())
print("Missing active values:", df_clean["is_active"].isnull().sum())

Missing reviews: 0
Missing active values: 0


In [21]:
null_percentages = df_clean.isnull().mean()

columns_to_drop = df_clean.columns[
    null_percentages > 0.50
].tolist()

print("Columns being dropped:", columns_to_drop)

df_clean = df_clean.drop(
    columns=columns_to_drop
)

print("Columns remaining:")
print(df_clean.columns.tolist())

Columns being dropped: ['weight_kg_duplicate']
Columns remaining:
['employee_id', 'name', 'age', 'salary', 'join_date', 'department', 'gender', 'country', 'city', 'weight_kg', 'is_active', 'review', 'price', 'target']


In [22]:
df_clean["employee_id"] = pd.to_numeric(
    df_clean["employee_id"],
    errors="raise"
).astype("int64")

df_clean["age"] = (
    df_clean["age"]
    .round()
    .astype("int64")
)

df_clean["salary"] = df_clean["salary"].astype(
    "float64"
)

df_clean["weight_kg"] = df_clean["weight_kg"].astype(
    "float64"
)

df_clean["price"] = df_clean["price"].astype(
    "float64"
)

df_clean["join_date"] = pd.to_datetime(
    df_clean["join_date"]
)

df_clean["is_active"] = df_clean["is_active"].astype(
    bool
)

df_clean["target"] = pd.to_numeric(
    df_clean["target"],
    errors="raise"
).astype("int64")

print(df_clean.dtypes)

employee_id             int64
name           string[python]
age                     int64
salary                float64
join_date      datetime64[ns]
department             object
gender                 object
country                object
city                   object
weight_kg             float64
is_active                bool
review         string[python]
price                 float64
target                  int64
dtype: object


In [23]:
expected_columns = [
    "employee_id",
    "name",
    "age",
    "salary",
    "join_date",
    "department",
    "gender",
    "country",
    "city",
    "weight_kg",
    "is_active",
    "review",
    "price",
    "target"
]

# Validate schema
assert list(df_clean.columns) == expected_columns

# Validate missing values
assert df_clean.isnull().sum().sum() == 0

# Validate duplicates
assert df_clean.duplicated().sum() == 0
assert df_clean["employee_id"].duplicated().sum() == 0

# Validate ranges
assert df_clean["age"].between(18, 65).all()
assert df_clean["salary"].between(15000, 500000).all()
assert (df_clean["price"] > 0).all()
assert (df_clean["price"] <= 10000).all()

# Validate labels
assert set(df_clean["gender"].unique()) <= {
    "Male",
    "Female"
}

assert set(df_clean["country"].unique()) <= {
    "Bangladesh",
    "United Kingdom",
    "United States"
}

print("All validation tests passed successfully.")

print("\nFinal shape:", df_clean.shape)
print("Missing values:", df_clean.isnull().sum().sum())
print("Exact duplicates:", df_clean.duplicated().sum())
print(
    "Duplicate employee IDs:",
    df_clean["employee_id"].duplicated().sum()
)

All validation tests passed successfully.

Final shape: (19801, 14)
Missing values: 0
Exact duplicates: 0
Duplicate employee IDs: 0


In [24]:
display(df_clean.head(10))

print("\nCleaned dataset information:")
df_clean.info()

print("\nNumerical summary:")
display(df_clean.describe())

,employee_id,name,age,salary,join_date,department,gender,country,city,weight_kg,is_active,review,price,target
0,12369,Hasan D.,42,35149.0,2023-05-30,Sales,Male,United States,Sylhet,118.4,False,Good value for money.,219.18,0
1,7361,Grace P.,42,77836.0,2017-02-27,Engineering,Female,United States,Chittagong,71.0,True,Product met all expectations.,329.26,0
2,3446,Eve D.,20,35404.0,2022-01-18,HR,Male,Bangladesh,Chittagong,98.7,False,Very satisfied with the quality.,438.60,0
3,18416,Diana E.,47,107375.0,2017-04-15,Marketing,Male,United Kingdom,Khulna,86.6,False,Delivery was fast and packaging great.,238.03,0
4,13677,Ira W.,64,55962.0,2021-11-02,Sales,Female,United Kingdom,Dhaka,83.5,False,Would buy again without hesitation.,9.40,0
5,458,Frank Y.,38,88527.0,2022-08-16,Marketing,Male,United Kingdom,Sylhet,95.8,True,"Excellent product, highly recommend!",213.25,0
6,16484,John Q.,25,109079.0,2020-09-10,HR,Female,United States,Chittagong,125.2,False,Delivery was fast and packaging great.,19.35,0
7,12951,Charlie K.,22,91118.0,2010-01-12,Finance,Female,United Kingdom,Rajshahi,87.9,False,Delivery was fast and packaging great.,419.00,0
8,13108,Diana V.,43,88859.0,2016-07-21,Marketing,Male,United States,Khulna,73.8,True,No Review,430.17,0
9,14925,Alice C.,42,46627.0,2012-01-12,Sales,Female,United States,Khulna,110.7,False,Very satisfied with the quality.,429.27,0



Cleaned dataset information:
<class 'pandas.core.frame.DataFrame'>
Index: 19801 entries, 0 to 20299
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   employee_id  19801 non-null  int64         
 1   name         19801 non-null  string        
 2   age          19801 non-null  int64         
 3   salary       19801 non-null  float64       
 4   join_date    19801 non-null  datetime64[ns]
 5   department   19801 non-null  object        
 6   gender       19801 non-null  object        
 7   country      19801 non-null  object        
 8   city         19801 non-null  object        
 9   weight_kg    19801 non-null  float64       
 10  is_active    19801 non-null  bool          
 11  review       19801 non-null  string        
 12  price        19801 non-null  float64       
 13  target       19801 non-null  int64         
dtypes: bool(1), datetime64[ns](1), float64(3), int64(3), object(4), string(2)
mem

,employee_id,age,salary,join_date,weight_kg,price,target
count,19801.000000,19801.000000,19801.000000,19801,19801.000000,19801.000000,19801.000000
mean,10001.784203,41.493460,75974.667062,2017-07-07 00:18:50.124741376,86.994223,251.304189,0.049846
min,1.000000,18.000000,30000.000000,2010-01-01 00:00:00,45.000000,5.000000,0.000000
25%,5001.000000,30.000000,53358.000000,2013-12-08 00:00:00,66.900000,133.800000,0.000000
50%,10008.000000,42.000000,75126.000000,2017-06-30 00:00:00,86.600000,250.100000,0.000000
75%,15000.000000,53.000000,96606.000000,2021-01-28 00:00:00,107.200000,367.660000,0.000000
max,20000.000000,65.000000,165339.500000,2024-12-30 00:00:00,130.000000,499.970000,1.000000
std,5772.567720,13.522393,27444.080432,NaN,23.820407,139.598850,0.217632


In [25]:
target_count = df_clean["target"].value_counts()
target_percentage = (
    df_clean["target"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("Target counts:")
print(target_count)

print("\nTarget percentages:")
print(target_percentage)

Target counts:
target
0    18814
1      987
Name: count, dtype: int64

Target percentages:
target
0    95.02
1     4.98
Name: proportion, dtype: float64


In [26]:
output_path = "/content/cleaned_dataset.csv"

df_clean.to_csv(
    output_path,
    index=False,
    date_format="%Y-%m-%d"
)

print("Cleaned dataset created successfully.")
print("File location:", output_path)

Cleaned dataset created successfully.
File location: /content/cleaned_dataset.csv


In [27]:
saved_dataset = pd.read_csv(
    "/content/cleaned_dataset.csv"
)

print("Saved CSV shape:", saved_dataset.shape)
print(
    "Saved CSV missing values:",
    saved_dataset.isnull().sum().sum()
)

display(saved_dataset.head())

Saved CSV shape: (19801, 14)
Saved CSV missing values: 0


,employee_id,name,age,salary,join_date,department,gender,country,city,weight_kg,is_active,review,price,target
0,12369,Hasan D.,42,35149.0,2023-05-30,Sales,Male,United States,Sylhet,118.4,False,Good value for money.,219.18,0
1,7361,Grace P.,42,77836.0,2017-02-27,Engineering,Female,United States,Chittagong,71.0,True,Product met all expectations.,329.26,0
2,3446,Eve D.,20,35404.0,2022-01-18,HR,Male,Bangladesh,Chittagong,98.7,False,Very satisfied with the quality.,438.60,0
3,18416,Diana E.,47,107375.0,2017-04-15,Marketing,Male,United Kingdom,Khulna,86.6,False,Delivery was fast and packaging great.,238.03,0
4,13677,Ira W.,64,55962.0,2021-11-02,Sales,Female,United Kingdom,Dhaka,83.5,False,Would buy again without hesitation.,9.40,0


In [28]:
files.download(
    "/content/cleaned_dataset.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>